

# LSTM Hot-Cold Path Integration Report

## Overview
A complete LSTM trading system using real Coinbase L2 order book data with cold path (Python) for training and hot path (C++) for low-latency inference.

---

## 1. Training Features (51 total)

| Category | Features |
|----------|----------|
| **Price Returns** | 1, 2, 5, 10, 20 tick returns |
| **Momentum** | 5, 10, 20 tick momentum |
| **MA Ratios** | Price/MA ratios for 5, 10, 20 periods |
| **Spread** | Raw spread, spread in bps |
| **Imbalance** | Top-of-book imbalance, 5-level depth imbalance |
| **Microprice** | Volume-weighted microprice deviation from mid |
| **Order Flow** | OFI (order flow imbalance), bid/ask pressure |
| **Lagged Features** | Lagged versions of above (1, 2, 5 ticks) |

**Data Source**: Coinbase L2 order book NDJSON files (38 files, ~29 products)

---

## 2. Target Variable



In [ ]:
# Forward mid-price return (next N ticks)
target = (future_mid_price - current_mid_price) / current_mid_price
# Scaled and clipped to [-1, 1]



**Prediction**: Direction and magnitude of short-term price movement.

---

## 3. Model Architecture



In [ ]:
Input (51 features) → Input Projection (51 → 64)
    ↓
Bidirectional LSTM (2 layers, 64 hidden units)
    ↓
Attention Layer (weights important timesteps)
    ↓
FC Layers (128 → 64 → 32)
    ↓
Output (1 unit: predicted return)

## Model Save/Load Summary

### **Save (Python → Binary File)**



In [ ]:
# Export weights as flat float32 array
with open("model.bin", "wb") as f:
    # 1. Write 32-byte header
    f.write(struct.pack("4sIIIIIII", b"LSIM", 1, 51, 64, 2, 1, 1, 0))
    
    # 2. Write all weights as contiguous float32
    for param in model.parameters():
        f.write(param.detach().cpu().numpy().astype(np.float32).tobytes())
    
    # 3. Write normalization params (means, stds)
    f.write(feature_means.astype(np.float32).tobytes())
    f.write(feature_stds.astype(np.float32).tobytes())



### **Load (C++ via Memory-Map)**



In [ ]:
// Memory-map the file (zero-copy)
int fd = open("model.bin", O_RDONLY);
void* data = mmap(nullptr, file_size, PROT_READ, MAP_PRIVATE, fd, 0);

// Read header
auto* header = (ModelHeader*)data;

// Point directly to weights in mapped memory
const float* weights = (float*)((char*)data + 32);

// Copy to aligned buffers for inference
std::copy(weights, weights + N, input_weights_.begin());



**Key idea**: Python flattens all tensors to a single binary blob → C++ memory-maps it and reads weights at known offsets. No serialization library needed.



---

## 4. Model Export Format (Cold → Hot)

### Binary File: `lstm_trading_simple.bin` (753 KB)



In [ ]:
┌─────────────────────────────────────────┐
│ HEADER (32 bytes)                       │
│ - Magic: "LSIM" (4 bytes)               │
│ - Version: 1 (4 bytes)                  │
│ - input_size: 51 (4 bytes)              │
│ - hidden_size: 64 (4 bytes)             │
│ - num_layers: 2 (4 bytes)               │
│ - output_size: 1 (4 bytes)              │
│ - flags: bidirectional=1 (4 bytes)      │
│ - reserved (4 bytes)                    │
├─────────────────────────────────────────┤
│ WEIGHTS (flat float32 array)            │
│ - Input projection: W, b                │
│ - LSTM layer 0 fwd: Wi, Wh, bi, bh      │
│ - LSTM layer 0 bwd: Wi, Wh, bi, bh      │
│ - LSTM layer 1 fwd: Wi, Wh, bi, bh      │
│ - LSTM layer 1 bwd: Wi, Wh, bi, bh      │
│ - Attention: W, b                       │
│ - FC layers: fc1, fc2, fc3, output      │
├─────────────────────────────────────────┤
│ NORMALIZATION PARAMS                    │
│ - feature_means[51]                     │
│ - feature_stds[51]                      │
└─────────────────────────────────────────┘



### Config File: `lstm_trading_simple.json`


In [ ]:
{
  "input_size": 51,
  "hidden_size": 64,
  "num_layers": 2,
  "bidirectional": true,
  "sequence_length": 20,
  "feature_names": ["return_1", "return_2", ...]
}



---

## 5. Hot Path Loading (C++)



In [ ]:
// Memory-mapped file for zero-copy loading
class MappedFile {
    void* data_;  // mmap'd pointer
    size_t size_;
};

// Load weights directly from mmap
void load(const std::string& path) {
    mapped_file_.open(path);
    auto* header = reinterpret_cast<const ModelHeader*>(mapped_file_.data());
    
    // Weights pointer directly into mmap'd memory (zero-copy)
    const float* weights = reinterpret_cast<const float*>(
        static_cast<const char*>(mapped_file_.data()) + sizeof(ModelHeader)
    );
    
    // Copy to cache-aligned buffers for SIMD
    input_proj_w_.assign(weights, weights + input_size * hidden_size);
    // ... etc
}



**Key optimizations**:
- Memory-mapped I/O (no malloc/copy on load)
- Cache-aligned buffers (`alignas(64)`)
- Pre-allocated scratch space

---

## 6. Hot Path Inference (C++)



In [ ]:
// Real-time feature extraction (~500ns)
void update(double bid, double ask, double bid_qty, double ask_qty) {
    // Update rolling statistics
    // Compute all 51 features in-place
}

// LSTM forward pass (~17μs)
float predict(const float* features) {
    // 1. Normalize features
    // 2. Input projection
    // 3. LSTM layers (bidirectional)
    // 4. Attention pooling
    // 5. FC layers
    // 6. Return prediction
}



---

## 7. Performance Results

| Metric | Value |
|--------|-------|
| Feature extraction | ~500 ns/tick |
| LSTM inference | ~17,000 ns/tick |
| **Total latency** | **~18 μs/prediction** |
| Throughput | ~55,000 predictions/sec |
| Model load time | <1 ms (mmap) |

---

## 8. Integration with Trading Strategy



In [ ]:
// In strats/lstm_strategy.hpp
class LSTMStrategy : public Strategy {
    LSTMInference lstm_;
    FeatureExtractor features_;
    
    void on_book_update(const BookUpdate& update) override {
        features_.update(update.bid, update.ask, ...);
        
        if (features_.ready()) {
            float signal = lstm_.predict(features_.get_features());
            
            if (signal > threshold_) {
                send_buy_order(...);
            } else if (signal < -threshold_) {
                send_sell_order(...);
            }
        }
    }
};



---

## Files Created

| Path | Purpose |
|------|---------|
| l2_data_loader.py | Parse Coinbase L2 NDJSON |
| feature_extractor.py | Python feature extraction |
| lstm_model.py | PyTorch LSTM model |
| training_pipeline.py | Training loop with early stopping |
| export_model.py | Export to binary format |
| lstm_inference.hpp | C++ inference engine |
| feature_extractor.hpp | C++ real-time features |
| lstm_strategy.hpp | Strategy integration |
| lstm_trading_simple.bin | Exported weights |
| lstm_trading_simple.json | Model config |

---

## Summary

✅ **Cold Path**: Python trains LSTM on L2 order book data, exports to custom binary format  
✅ **Hot Path**: C++ loads via mmap, runs inference in ~18μs  
✅ **Integration**: Plugs into existing HFT strategy framework  
✅ **Latency**: Sub-20μs total prediction time suitable for HFT